# Sandbox

## Init

In [ ]:
import sys
sys.path.append('..')
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from tools import serialTools, captureTools, eval

In [ ]:
datasetsPath = '../datasets/gyro/' # Set location of dataset
gyro = pd.read_csv(datasetsPath + 'gyro_mobile.csv')
# Preprocessing dataframe
gyro = gyro.drop(columns='timestamp') # Drop timestamp column

xtrain, xtest, ytrain, ytest = train_test_split( # Split into training and test datasets
    gyro.iloc[:,:6],
    gyro.iloc[:,6:],
    test_size=0.2,
    random_state=0
)

# evaldata=[(xtrain,ytrain),(xtest,ytest)]          # Datensatz zur Evaluierung
evaldata=[(xtest,ytest)]          # Datensatz zur Evaluierung

In [ ]:
donor = XGBClassifier()
final = XGBClassifier()
bestIter = 0

def trainDonor():
    global bestIter
    # Training the donor model
    donor.set_params(
        objective='binary:logistic',
        n_estimators=10000,             # "Große Anzahl an Schaetzern, die nicht erreicht werden soll"
        early_stopping_rounds=20,       # Anzahl an Runden, bei denen sich das Modell nicht verbessern muss, bis abgebrochen wird
        max_depth=2,
        learning_rate=0.1
    )
    donor.fit(
        xtrain, 
        ytrain, 
        eval_set=evaldata, 
        verbose=False
    )
    bestIter = donor.best_iteration

def trainModel(model, bestIter, prints: bool = False):
    # Set default value, if bestIter hasn't been set yet
    if bestIter == 0:
        bestIter = 50
    
    model.set_params(
        objective='binary:logistic',
        # tree_method = 'exact',
        n_estimators=bestIter,
        max_depth=2,
        learning_rate=0.1,
        base_score=0.5
    )
    if prints == True:
        print(f'Model trained using {bestIter} estimators.')

    model.fit(xtrain, 
        ytrain, 
        eval_set=evaldata, 
        verbose=False
    )

## Amount of trees generated after training with **early stopping** vs **bestIter**
- The Donor model generated the "best" amount of trees + the amount of early stopping rounds
- The final model is actually smaller than the donor but also lost a small amount of accuracy in this case

In [ ]:
trainDonor()
trainModel(final, bestIter)

print(accuracy_score(ytest, donor.predict(xtest))) # Accuracy Score: 0.9835911861228317
dump_list = donor.get_booster().get_dump()
num_trees = len(dump_list)
print(num_trees) # Number of trees: 376

print(accuracy_score(ytest, final.predict(xtest))) # Accuracy Score: 0.9831223628691983
dump_list = final.get_booster().get_dump()
num_trees = len(dump_list)
print(num_trees) # Number of trees: 355